## Exercise: Text Preprocessing with Moby Dick (moby.txt)

For this exercise, you will load text from the `moby.txt` file available in your Colab environment and apply the text preprocessing techniques we've learned using Keras Tokenizer and `pad_sequences`.

### Task 1: Loading the Text Data

In [2]:
# Path to the moby.txt file
file_path = "/content/moby.txt"

# Read the content of the file
with open(file_path, 'r', encoding='utf-8') as f:
    moby_text = f.read()

# Display the first 500 characters to verify
print(moby_text[:500])

[Moby Dick by Herman Melville 1851]


ETYMOLOGY.

(Supplied by a Late Consumptive Usher to a Grammar School)

The pale Usher--threadbare in coat, heart, body, and brain; I see him
now.  He was ever dusting his old lexicons and grammars, with a queer
handkerchief, mockingly embellished with all the gay flags of all the
known nations of the world.  He loved to dust his old grammars; it
somehow mildly reminded him of his mortality.

"While you take in hand to school others, and to teach them by wha


### Task 2: Preprocessing the Text

Apply tokenization, convert to sequences, and pad the sequences. Consider how you might split the large text into manageable 'sentences' or paragraphs for tokenization.

In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [4]:
# Split the text into lines/paragraphs as a proxy for sentences
# Filter out empty lines
moby_sentences = [line.strip() for line in moby_text.split('\n') if line.strip()]


In [5]:
# Initialize Tokenizer
# Consider a reasonable num_words and oov_token
# For this large text, you might want to increase num_words significantly
# Or, set it to None to tokenize all words, but be mindful of memory usage.

tokenizer_moby = Tokenizer(num_words=20000,
                           oov_token='<unk>') # Increased num_words for larger text


In [6]:

# Fit on the moby_sentences
tokenizer_moby.fit_on_texts(moby_sentences)


In [7]:

# Display some info about the tokenizer
print(f"Total unique words found: {len(tokenizer_moby.word_index)}")


Total unique words found: 17690


In [8]:
print("Top 10 words and their indices:")
for word, index in list(tokenizer_moby.word_index.items())[:10]:
    print(f"  {word}: {index}")


Top 10 words and their indices:
  <unk>: 1
  the: 2
  of: 3
  and: 4
  a: 5
  to: 6
  in: 7
  that: 8
  his: 9
  it: 10


In [9]:

# Convert text to sequences
moby_sequences = tokenizer_moby.texts_to_sequences(moby_sentences)


In [10]:

# Determine an appropriate maxlen for padding
# A common approach is to use a percentile of sentence lengths
sequence_lengths = [len(seq) for seq in moby_sequences if len(seq) > 0]

if sequence_lengths:
    median_length = int(np.median(sequence_lengths))
    # Cap maxlen at a reasonable value like 100 or 2x median, adjust as needed
    max_sequence_length = min(median_length * 2 + 5, 100) # Added a small buffer
else:
    max_sequence_length = 50 # Default if no sequences

print(f"\nMedian sequence length: {median_length if sequence_lengths else 'N/A'}")



Median sequence length: 12


In [11]:
print(f"Chosen maxlen for padding: {max_sequence_length}")

# Pad the sequences
padded_moby_sequences = pad_sequences(moby_sequences,
                                         maxlen=max_sequence_length,
                                         padding='post',
                                         truncating='post')


Chosen maxlen for padding: 29


In [12]:

# Display the first few padded sequences
print("\nFirst 5 padded sequences (truncated to 10 elements for display):")
for i, seq in enumerate(padded_moby_sequences[:5]):
    print(f"  Sequence {i+1}: {seq[:10]}...")


First 5 padded sequences (truncated to 10 elements for display):
  Sequence 1: [ 284  287   22 9762 9763 5155    0    0    0    0]...
  Sequence 2: [9764    0    0    0    0    0    0    0    0    0]...
  Sequence 3: [1672   22    5  708 9765 6833    6    5 6834 2017]...
  Sequence 4: [   2 1070 6833 9766    7  754  257  229    4  598]...
  Sequence 5: [  33   13   17  137 6835    9   60 9767    4 6836]...


### Task 3: Critical Thinking Questions

Answer the following questions based on your preprocessing steps:

1.  **Tokenizer Parameters:** How did your choice of `num_words` and `oov_token` affect the `word_index` and subsequent `moby_sequences`? What are the trade-offs of setting `num_words` to a very high value versus a lower one, especially for a large text like Moby Dick?
2.  **Sequence Length and Padding:** How did you determine the `maxlen` for `pad_sequences`? What happens if `maxlen` is too small or too large? Describe the impact of `padding='pre'` vs `padding='post'` and `truncating='pre'` vs `truncating='post'` in the context of sequence data for machine learning models.
3.  **Impact of Standardization:** The default `standardize="lower_and_strip_punctuation"` was used implicitly. How might different standardization rules (e.g., keeping punctuation, not converting to lowercase) affect the tokenization process and the resulting `word_index`? Provide an example of a scenario where you might want to customize standardization for a literary text like Moby Dick.
4.  **Beyond Words:** The `Tokenizer` defaults to word-level tokenization. When might character-level tokenization or n-gram tokenization be more appropriate for analyzing a text like Moby Dick? What are the challenges and benefits of each approach?

### Task 3: Critical Thinking Questions Answers

#### 1. Tokenizer Parameters:

*   **`num_words` and `oov_token` effect:** We set `num_words=20000`. Since Moby Dick had 17,690 unique words, all of them were included in our `word_index`, and thus in `moby_sequences`. The `oov_token='<unk>'` was ready for any words not in our text, but wasn't actually used here because `num_words` was high enough.

*   **Trade-offs of `num_words` (high vs. low):**
    *   **High `num_words`**: Captures almost all words, preserving rich detail (great for literary texts!) but makes the vocabulary (and model) larger, potentially slower, and more memory-intensive.
    *   **Low `num_words`**: Reduces vocabulary size, saving memory and speeding up training. However, it means many words are replaced by `<unk>`, leading to significant information loss, which can hurt model performance.

#### 2. Sequence Length and Padding:

*   **Determining `maxlen`**: We picked `maxlen=29`. This was based on the median sequence length (12) multiplied by two, plus a small buffer, then capped at 100. This balances capturing most sentence information without making sequences excessively long.

*   **Impact of `maxlen` being too small or too large:**
    *   **Too small**: You lose important information because sequences are truncated too much, hurting the model's understanding.
    *   **Too large**: Wastes memory and computation because of too much padding (zeros), making training inefficient.

*   **`padding='pre'` vs `padding='post'` and `truncating='pre'` vs `truncating='post'`:**
    *   **`padding='pre'`**: Adds zeros to the *start* of sequences. Useful for RNNs that process left-to-right, as key info arrives later.
    *   **`padding='post'`**: Adds zeros to the *end* of sequences. More common and often intuitive, keeping the main content at the start.
    *   **`truncating='pre'`**: Chops off the *start* of long sequences. Keeps the end, which might be more critical for some tasks.
    *   **`truncating='post'`**: Chops off the *end* of long sequences. We used this; it keeps the beginning of the sentence, often containing crucial context.

#### 3. Impact of Standardization:

*   **`standardize="lower_and_strip_punctuation"` (implicit default)**: This made everything lowercase and removed punctuation. So, 'Moby', 'moby.', and 'moby' all became just 'moby'. This simplifies the vocabulary but loses some detail.

*   **How different standardization rules might affect `word_index`:**
    *   **Keeping punctuation**: `word.` would be different from `word`, and punctuation marks themselves would be tokens. This means a larger vocabulary.
    *   **Not converting to lowercase**: 'Moby' and 'moby' would be distinct words. This would drastically inflate the `word_index` size, as capitalized words would have their own entries.

*   **Scenario for customizing standardization for Moby Dick:**
    *   **Scenario**: If we wanted to analyze Melville's specific use of capitalization (e.g., 'Ahab' vs. 'ahab') or punctuation to understand literary style or character nuances. Simply lowercasing and stripping punctuation would destroy this information.
    *   **Customization**: We could use a custom `standardize` function that preserves capitalization for proper nouns (perhaps using a predefined list or NER) and keeps specific punctuation that's important for stylistic analysis (like hyphens or quotation marks).

#### 4. Beyond Words:

*   **Character-level tokenization:**
    *   **When appropriate for Moby Dick**: Good for analyzing morphology, sub-word units, or handling misspellings/archaic words. Also useful for generating text in a specific style.
    *   **Challenges**: Creates very long sequences, increasing computational cost. Characters alone carry less meaning, making it harder for models to learn complex ideas.
    *   **Benefits**: Handles unknown words well (if characters are known). Robust to typos and useful for low-level linguistic analysis or generation.

*   **N-gram tokenization:**
    *   **When appropriate for Moby Dick**: Useful for finding recurring phrases, idioms, or unique multi-word expressions (like "white whale" or "call me Ishmael"). Great for stylistic analysis, authorship attribution, or topic modeling where specific phrases are key.
    *   **Challenges**: Leads to a huge, sparse vocabulary, making it expensive and memory-intensive. Doesn't capture long-range dependencies well.
    *   **Benefits**: Explicitly captures local word context and phrase meaning. Strong for tasks like sentiment analysis or identifying common collocations.